# Risk Analysis: VaR & Monte Carlo Simulation

**Category:** 11-Financial  
**Description:** Value at Risk, stress testing, Monte Carlo methods  
**Libraries:** NumPy, Pandas, SciPy, Matplotlib

---

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q numpy pandas scipy matplotlib

CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
MODEL = CLAUDE

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

---

# Part 1: Generate Portfolio Data

---

In [ ]:
# Simulate a diversified portfolio
n_days = 252 * 5  # 5 years
portfolio_value = 1_000_000  # $1M portfolio

# Asset parameters (annual)
assets = {
    'US_Equities': {'weight': 0.40, 'return': 0.10, 'vol': 0.18},
    'Intl_Equities': {'weight': 0.20, 'return': 0.08, 'vol': 0.22},
    'Bonds': {'weight': 0.25, 'return': 0.04, 'vol': 0.05},
    'REITs': {'weight': 0.10, 'return': 0.07, 'vol': 0.20},
    'Commodities': {'weight': 0.05, 'return': 0.03, 'vol': 0.25}
}

# Correlation matrix
corr = np.array([
    [1.00, 0.70, 0.10, 0.60, 0.30],
    [0.70, 1.00, 0.05, 0.50, 0.40],
    [0.10, 0.05, 1.00, 0.20, 0.00],
    [0.60, 0.50, 0.20, 1.00, 0.35],
    [0.30, 0.40, 0.00, 0.35, 1.00]
])

# Generate correlated returns
vols = np.array([a['vol'] for a in assets.values()]) / np.sqrt(252)
means = np.array([a['return'] for a in assets.values()]) / 252
weights = np.array([a['weight'] for a in assets.values()])

cov = np.outer(vols, vols) * corr
L = np.linalg.cholesky(cov)

random_returns = np.random.randn(n_days, len(assets))
asset_returns = means + random_returns @ L.T

# Portfolio daily returns
portfolio_returns = asset_returns @ weights

# Create DataFrame
dates = pd.date_range('2020-01-01', periods=n_days, freq='B')
returns_df = pd.DataFrame(asset_returns, index=dates, columns=assets.keys())
returns_df['Portfolio'] = portfolio_returns

print(f"Portfolio Statistics (Daily):")
print(f"  Mean Return: {portfolio_returns.mean()*252:.2%} (annualized)")
print(f"  Volatility:  {portfolio_returns.std()*np.sqrt(252):.2%} (annualized)")
print(f"  Sharpe:      {portfolio_returns.mean()*252 / (portfolio_returns.std()*np.sqrt(252)):.2f}")

In [ ]:
# Plot portfolio evolution
portfolio_values = portfolio_value * (1 + returns_df['Portfolio']).cumprod()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Portfolio value
axes[0, 0].plot(portfolio_values, 'b-', linewidth=1)
axes[0, 0].axhline(portfolio_value, color='gray', linestyle='--', label='Initial')
axes[0, 0].set_title('Portfolio Value Over Time')
axes[0, 0].set_ylabel('Value ($)')
axes[0, 0].legend()
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Returns distribution
axes[0, 1].hist(portfolio_returns * 100, bins=50, density=True, alpha=0.7, color='steelblue')
x = np.linspace(-5, 5, 100)
axes[0, 1].plot(x, stats.norm.pdf(x, portfolio_returns.mean()*100, portfolio_returns.std()*100), 
                'r-', linewidth=2, label='Normal fit')
axes[0, 1].set_title('Daily Returns Distribution')
axes[0, 1].set_xlabel('Daily Return (%)')
axes[0, 1].legend()

# Drawdown
rolling_max = portfolio_values.cummax()
drawdown = (portfolio_values - rolling_max) / rolling_max
axes[1, 0].fill_between(drawdown.index, drawdown * 100, 0, alpha=0.5, color='red')
axes[1, 0].set_title('Drawdown')
axes[1, 0].set_ylabel('Drawdown (%)')
axes[1, 0].set_ylim(-30, 0)

# Rolling volatility
rolling_vol = returns_df['Portfolio'].rolling(21).std() * np.sqrt(252) * 100
axes[1, 1].plot(rolling_vol, 'purple', linewidth=1)
axes[1, 1].axhline(rolling_vol.mean(), color='gray', linestyle='--')
axes[1, 1].set_title('21-Day Rolling Volatility')
axes[1, 1].set_ylabel('Annualized Vol (%)')

plt.tight_layout()
plt.show()

print(f"\nMax Drawdown: {drawdown.min():.1%}")

---

# Part 2: Value at Risk (VaR)

---

In [ ]:
def calculate_var(returns, confidence=0.95, method='historical', portfolio_value=1_000_000):
    """
    Calculate Value at Risk.
    
    Args:
        returns: Array of returns
        confidence: Confidence level (e.g., 0.95 for 95%)
        method: 'historical', 'parametric', or 'cornish_fisher'
    
    Returns:
        VaR in dollars
    """
    if method == 'historical':
        var_pct = np.percentile(returns, (1 - confidence) * 100)
    
    elif method == 'parametric':
        # Assume normal distribution
        z = stats.norm.ppf(1 - confidence)
        var_pct = returns.mean() + z * returns.std()
    
    elif method == 'cornish_fisher':
        # Adjust for skewness and kurtosis
        z = stats.norm.ppf(1 - confidence)
        s = stats.skew(returns)
        k = stats.kurtosis(returns)
        
        z_cf = (z + (z**2 - 1) * s / 6 + 
                (z**3 - 3*z) * (k - 3) / 24 - 
                (2*z**3 - 5*z) * s**2 / 36)
        var_pct = returns.mean() + z_cf * returns.std()
    
    return -var_pct * portfolio_value

# Calculate VaR at different confidence levels
confidence_levels = [0.90, 0.95, 0.99]

print("="*60)
print("VALUE AT RISK (1-Day)")
print("="*60)
print(f"Portfolio Value: ${portfolio_value:,.0f}")
print()

for conf in confidence_levels:
    var_hist = calculate_var(portfolio_returns, conf, 'historical', portfolio_value)
    var_param = calculate_var(portfolio_returns, conf, 'parametric', portfolio_value)
    var_cf = calculate_var(portfolio_returns, conf, 'cornish_fisher', portfolio_value)
    
    print(f"{conf:.0%} Confidence:")
    print(f"  Historical:     ${var_hist:>10,.0f}")
    print(f"  Parametric:     ${var_param:>10,.0f}")
    print(f"  Cornish-Fisher: ${var_cf:>10,.0f}")
    print()

In [ ]:
# Visualize VaR
var_95 = calculate_var(portfolio_returns, 0.95, 'historical', portfolio_value)
var_99 = calculate_var(portfolio_returns, 0.99, 'historical', portfolio_value)

plt.figure(figsize=(12, 6))
plt.hist(portfolio_returns * portfolio_value, bins=50, density=True, alpha=0.7, 
         color='steelblue', label='P&L Distribution')

plt.axvline(-var_95, color='orange', linewidth=2, linestyle='--', 
            label=f'95% VaR: ${var_95:,.0f}')
plt.axvline(-var_99, color='red', linewidth=2, linestyle='--', 
            label=f'99% VaR: ${var_99:,.0f}')
plt.axvline(0, color='gray', linewidth=1)

plt.title('Daily P&L Distribution with VaR')
plt.xlabel('Daily P&L ($)')
plt.ylabel('Density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
%%ai $MODEL
Explain Value at Risk (VaR) for a risk manager:

1. What does "95% VaR of $15,000" actually mean?
2. What are the limitations of VaR?
3. Why do we use multiple methods (historical, parametric, Cornish-Fisher)?
4. What is Expected Shortfall (CVaR) and why is it sometimes preferred?

---

# Part 3: Monte Carlo VaR

---

In [ ]:
def monte_carlo_var(returns_df, weights, n_simulations=10000, n_days=1, 
                    confidence=0.95, portfolio_value=1_000_000):
    """
    Monte Carlo simulation for VaR.
    """
    # Estimate parameters
    asset_returns = returns_df.drop('Portfolio', axis=1)
    mean_returns = asset_returns.mean().values
    cov_matrix = asset_returns.cov().values
    
    # Simulate
    L = np.linalg.cholesky(cov_matrix)
    simulated_returns = np.zeros(n_simulations)
    
    for i in range(n_simulations):
        # Simulate n_days of returns
        random_returns = np.random.randn(n_days, len(weights))
        daily_returns = mean_returns + random_returns @ L.T
        
        # Compound returns over n_days
        cumulative_return = np.prod(1 + daily_returns @ weights) - 1
        simulated_returns[i] = cumulative_return
    
    # Calculate VaR
    var_pct = np.percentile(simulated_returns, (1 - confidence) * 100)
    var_dollar = -var_pct * portfolio_value
    
    # Calculate Expected Shortfall (CVaR)
    es_pct = simulated_returns[simulated_returns <= var_pct].mean()
    es_dollar = -es_pct * portfolio_value
    
    return simulated_returns, var_dollar, es_dollar

# Run simulation
sim_returns, var_mc, es_mc = monte_carlo_var(returns_df, weights, n_simulations=50000)

print(f"Monte Carlo VaR (95%, 1-Day): ${var_mc:,.0f}")
print(f"Expected Shortfall (CVaR):   ${es_mc:,.0f}")

In [ ]:
# Multi-day VaR
holding_periods = [1, 5, 10, 21]  # Days

print("\nMulti-Day VaR (95% Confidence):")
print("-" * 40)

for days in holding_periods:
    _, var, es = monte_carlo_var(returns_df, weights, n_simulations=10000, 
                                  n_days=days, confidence=0.95)
    print(f"  {days:2d}-Day VaR: ${var:>12,.0f} | ES: ${es:>12,.0f}")

In [ ]:
# Visualize Monte Carlo simulation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(sim_returns * portfolio_value, bins=100, density=True, alpha=0.7, color='steelblue')
axes[0].axvline(-var_mc, color='orange', linewidth=2, label=f'95% VaR: ${var_mc:,.0f}')
axes[0].axvline(-es_mc, color='red', linewidth=2, label=f'ES (CVaR): ${es_mc:,.0f}')
axes[0].set_title('Monte Carlo P&L Distribution (50,000 simulations)')
axes[0].set_xlabel('P&L ($)')
axes[0].legend()

# Tail focus
tail_returns = sim_returns[sim_returns < np.percentile(sim_returns, 10)]
axes[1].hist(tail_returns * portfolio_value, bins=50, alpha=0.7, color='red')
axes[1].axvline(-var_mc, color='orange', linewidth=2, linestyle='--', label=f'95% VaR')
axes[1].axvline(-es_mc, color='darkred', linewidth=2, linestyle='--', label=f'ES')
axes[1].set_title('Left Tail (Worst 10% of outcomes)')
axes[1].set_xlabel('P&L ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

---

# Part 4: Stress Testing

---

In [ ]:
# Historical stress scenarios
stress_scenarios = {
    '2008 Financial Crisis': {'US_Equities': -0.40, 'Intl_Equities': -0.45, 
                               'Bonds': 0.05, 'REITs': -0.35, 'Commodities': -0.30},
    'COVID Crash (Mar 2020)': {'US_Equities': -0.34, 'Intl_Equities': -0.32, 
                                'Bonds': 0.03, 'REITs': -0.40, 'Commodities': -0.25},
    'Dot-Com Bust (2000-02)': {'US_Equities': -0.45, 'Intl_Equities': -0.40, 
                                'Bonds': 0.15, 'REITs': 0.05, 'Commodities': 0.00},
    'Rising Rates Shock': {'US_Equities': -0.15, 'Intl_Equities': -0.20, 
                            'Bonds': -0.10, 'REITs': -0.25, 'Commodities': 0.05},
    'Stagflation': {'US_Equities': -0.20, 'Intl_Equities': -0.25, 
                     'Bonds': -0.15, 'REITs': -0.20, 'Commodities': 0.20}
}

print("="*60)
print("STRESS TEST RESULTS")
print("="*60)
print(f"Portfolio Value: ${portfolio_value:,}")
print()

stress_results = []
for scenario_name, shocks in stress_scenarios.items():
    shock_returns = np.array([shocks[asset] for asset in assets.keys()])
    portfolio_loss = (shock_returns * weights).sum() * portfolio_value
    stress_results.append({'Scenario': scenario_name, 'Loss': portfolio_loss})
    print(f"{scenario_name}:")
    print(f"  Portfolio Loss: ${-portfolio_loss:>12,.0f} ({(shock_returns * weights).sum():.1%})")
    print()

stress_df = pd.DataFrame(stress_results)

In [ ]:
# Visualize stress test results
fig, ax = plt.subplots(figsize=(12, 6))

scenarios = stress_df['Scenario']
losses = -stress_df['Loss'] / 1000  # Convert to thousands

colors = ['red' if l > 200 else 'orange' if l > 100 else 'yellow' for l in losses]
bars = ax.barh(scenarios, losses, color=colors, edgecolor='black')

ax.axvline(var_95/1000, color='blue', linestyle='--', linewidth=2, 
           label=f'95% VaR: ${var_95/1000:.0f}K')

ax.set_title('Portfolio Losses Under Stress Scenarios')
ax.set_xlabel('Loss ($K)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, loss in zip(bars, losses):
    ax.text(loss + 5, bar.get_y() + bar.get_height()/2, 
            f'${loss:.0f}K', va='center')

plt.tight_layout()
plt.show()

In [ ]:
%%ai $MODEL
Based on the stress test results:

1. Which scenario poses the greatest risk to this portfolio?
2. How could we modify the portfolio allocation to reduce crisis risk?
3. What's the relationship between VaR and stress test losses?
4. Should risk managers rely more on VaR or stress testing?

---

## Summary

This notebook covered:
- Portfolio risk metrics and drawdown analysis
- Value at Risk (VaR) calculation methods
- Expected Shortfall (CVaR)
- Monte Carlo simulation for risk estimation
- Historical stress testing scenarios

**Next:** Explore technical analysis in `04-technical-analysis.ipynb`

---